# 16: FD001 asymmetric objective experiment

Este notebook prueba si vale la pena entrenar LGBM con una objective asimetrica tipo C-MAPSS, penalizando mas la sobreestimacion de RUL y ponderando mas errores con RUL bajo.

Usa solo validacion artificial. No lee ni usa test oficial.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results" / "FD001"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
VALIDATION_RANDOM_STATE = 42
CUT_RULS = (20, 50, 80, 110, 140)
WINDOW_SIZE = 50
RUL_CAP = 125

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [ ]:
from src.preprocessed_FD001 import prepare_fd001_temporal_validation_only
from src.fd001_experiment_utils import (
    fit_predict_model,
    lgbm_cmapss_objective_factory,
    metric_row_from_predictions,
    selection_sort,
)


## Preparacion de validacion artificial


In [ ]:
base_prepared = prepare_fd001_temporal_validation_only(
    data_dir=DATA_DIR,
    eval_size=EVAL_SIZE,
    random_state=VALIDATION_RANDOM_STATE,
    max_rul=None,
    cut_ruls=CUT_RULS,
    window_size=WINDOW_SIZE,
)
prepared = dict(base_prepared)
prepared["y_train"] = base_prepared["y_train_raw"].clip(upper=RUL_CAP).copy()
representation = f"temporal_w{WINDOW_SIZE}"
print(prepared["X_train"].shape, prepared["X_eval"].shape)


## Entrenamiento con objective asimetrica


In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    asymmetric_predictions = fit_predict_model(
        prepared,
        lgbm_cmapss_objective_factory(random_state=VALIDATION_RANDOM_STATE),
        "LGBMRegressor_cmapss_weighted_objective",
        representation,
    )

asymmetric_predictions["sample_weight_scheme"] = "objective_rul_weighted"
asymmetric_row, asymmetric_bins = metric_row_from_predictions(
    asymmetric_predictions,
    extra={
        "model_name": "LGBMRegressor_cmapss_weighted_objective",
        "representation": representation,
        "window_size": WINDOW_SIZE,
        "rul_cap": RUL_CAP,
        "sample_weight_scheme": "objective_rul_weighted",
        "training_objective": "custom_cmapss_weighted",
        "target_used_for_training": "RUL_capped",
        "n_features": len(prepared["feature_columns"]),
    },
)
asymmetric_metrics = pd.DataFrame([asymmetric_row])
display(asymmetric_metrics)


## Comparacion contra LGBM actual


In [ ]:
sample_weight_results = pd.read_csv(RESULTS_DIR / "fd001_lgbm_sample_weight_search.csv")
comparison = pd.concat(
    [
        sample_weight_results.loc[
            sample_weight_results["sample_weight_scheme"].isin(["none", "bin_weights"])
        ].assign(training_objective="squared_error"),
        asymmetric_metrics,
    ],
    ignore_index=True,
    sort=False,
)
comparison = selection_sort(comparison)
comparison.to_csv(RESULTS_DIR / "fd001_lgbm_asymmetric_objective_metrics.csv", index=False)
asymmetric_predictions.to_csv(RESULTS_DIR / "fd001_lgbm_asymmetric_objective_predictions.csv", index=False)
asymmetric_bins.to_csv(RESULTS_DIR / "fd001_lgbm_asymmetric_objective_metrics_by_rul_bin.csv", index=False)

display(comparison[[
    "model_name",
    "representation",
    "window_size",
    "rul_cap",
    "sample_weight_scheme",
    "training_objective",
    "mae",
    "rmse",
    "r2",
    "cmapss_score",
    "cmapss_score_mean",
    "dangerous_error_pct",
    "mae_rul_0_30",
    "mae_rul_60_90",
]])


## Lectura esperada

Si la objective asimetrica baja C-MAPSS y dangerous_error_pct sin romper RMSE/MAE, vale la pena seguirla. Si empeora mucho score o RMSE, queda como experimento descartado o mejora futura.
